# 06 - Vision Module: Hiểu Ảnh, Không Tự Quyết Định Ý Định

Notebook này thay đổi góc nhìn quan trọng: **VLM là đôi mắt, router mới là người điều phối**.

VLM chỉ trả lời “trong ảnh có gì?”. `app/core/intent.py` mới kết hợp quan sát đó với câu người dùng để quyết định tìm sản phẩm, phối đồ, phân tích profile hay hỏi lại. Tách hai trách nhiệm giúp debug được lỗi do nhìn sai ảnh và lỗi do policy chọn sai route.


## BƯỚC 1: Setup Độc Lập

Notebook import implementation production thay vì chép lại prompt/hàm vision. Vì vậy kết quả debug phản ánh đúng code web đang chạy.


In [ ]:
import json
import sys
from pathlib import Path
from pprint import pprint

def find_project_root(start: Path | None = None) -> Path:
    '''Đi ngược cây thư mục cho tới khi thấy `app/core/vision.py`.'''
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "app" / "core" / "vision.py").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục Chatbot_Fashion")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.intent import route_user_request
from app.core.vision import (
    analyze_person_image,
    caption_product_image,
    describe_image_for_routing,
)
print(f"[OK] PROJECT_ROOT={PROJECT_ROOT}")


## BƯỚC 2: Bản Đồ Dữ Liệu Của Một Lượt Có Ảnh

```text
ảnh tạm + câu hỏi
       │
       ├─ câu hỏi đã rõ ───────────────► router chọn pipeline ngay
       │
       └─ câu hỏi trống/mơ hồ ─► VLM observation ─► router chọn hoặc hỏi lại
```

Ảnh raw chỉ tồn tại trong lúc xử lý. State dài hạn chỉ giữ mô tả có cấu trúc và candidate sản phẩm; không giữ base64 hay đường dẫn file tạm.


## BƯỚC 3: `describe_image_for_routing()` Trả Quan Sát Có Cấu Trúc

Output gồm `subject`, `caption`, `fashion_item`, `confidence`, `reason`. Không có trường `route`: đó là chủ ý thiết kế, không phải thiếu sót.


In [ ]:
def inspect_image_turn(image_path: str | Path, user_query: str = "") -> dict:
    '''Chạy VLM observation rồi đưa observation sang deterministic router.'''
    image_path = Path(image_path)
    if not image_path.exists():
        return {"status": "invalid_path", "path": str(image_path)}
    observation = describe_image_for_routing(str(image_path), user_query)
    decision = route_user_request(
        user_query,
        has_image=True,
        image_context=observation,
    )
    result = {
        "status": "ok",
        "observation": observation,
        "decision": decision.to_debug_dict(),
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


## BƯỚC 4: Policy Sau Khi Hiểu Ảnh

- Nhận ra món thời trang đủ chắc: tự tìm sản phẩm tương tự, đồng thời hỏi người dùng có muốn phối đồ.
- Chưa chắc món nào là trọng tâm: mô tả điều nhìn thấy rồi đưa ba lựa chọn rõ ràng.
- Câu text đã nói “phối đồ” hoặc “phân tích dáng”: không cần gọi VLM observation chỉ để phân loại lại ý định.


In [ ]:
MOCK_OBSERVATIONS = [
    {"subject":"product", "caption":"một áo blazer đen", "fashion_item":"áo blazer đen", "confidence":0.92},
    {"subject":"unclear", "caption":"một người đứng trước gương", "fashion_item":"", "confidence":0.30},
]
for observation in MOCK_OBSERVATIONS:
    decision = route_user_request("", has_image=True, image_context=observation)
    print("\n---")
    print(json.dumps(decision.to_debug_dict(), ensure_ascii=False, indent=2))


## BƯỚC 5: Phân Tích Người Chỉ Tạo Candidate

`analyze_person_image()` trả `dang_nguoi`, `tone_da`, `nhan_xet`. API chỉ lưu candidate tạm và hỏi xác nhận; `nhan_xet` không trở thành profile bền vững. Đây là cách tránh biến suy đoán VLM thành “sự thật” về người dùng.


In [ ]:
def preview_profile_candidate(image_path: str | Path) -> dict:
    '''Phân tích ảnh người nhưng tuyệt đối không ghi session/profile.'''
    raw = analyze_person_image(str(image_path))
    candidate = {
        key: raw.get(key)
        for key in ("dang_nguoi", "tone_da")
        if raw.get(key)
    }
    output = {"candidate": candidate, "comment": raw.get("nhan_xet", ""), "saved": False}
    print(json.dumps(output, ensure_ascii=False, indent=2))
    return output


## BƯỚC 6: Smoke Test Thật (Mặc Định Không Gọi Model)

Điền đường dẫn và bật flag. Output luôn in cả observation lẫn toàn bộ decision để phân biệt lỗi vision với lỗi routing.


In [ ]:
RUN_VISION_DEMO = False
TEST_IMAGE_PATH = PROJECT_ROOT / "tests" / "sample_images" / "bodyNu.jpg"
TEST_USER_QUERY = ""

if RUN_VISION_DEMO:
    vision_demo = inspect_image_turn(TEST_IMAGE_PATH, TEST_USER_QUERY)
else:
    print("[SKIP] Đặt RUN_VISION_DEMO=True để gọi VLM thật.")


## Kết Luận

Notebook 06 chỉ sở hữu **visual observation**. Business intent và route hữu hạn nằm ở notebook 08/`app/core/intent.py`; orchestration nằm ở notebook 09/API. Nếu VLM nhìn đúng nhưng route sai, sửa policy router. Nếu observation đã sai, sửa prompt/model vision.
